# Generate mock data

## Imports

In [ ]:
# Imports
import os
from pathlib import Path
import random
from datetime import datetime, timedelta


## Initialize

In [ ]:
# Initialize seed
random.seed(42) # Ensures consistency when seed is defined

# Define base path - where data will go
base = Path.cwd().parent / "data/raw"

# Confirm
print(base)


### Patient Plan

In [ ]:
# ─── PATIENT PLAN ───
# IRB 1: PAT-001 to PAT-010
#   EHR: all 10
#   CT:  PAT-001, PAT-003, PAT-004, PAT-006, PAT-008, PAT-009 (6 of 10)
#   MRI: none
#
# IRB 2: 4 overlap (PAT-002, PAT-005, PAT-007, PAT-010) + 6 new (PAT-011 to PAT-016)
#   EHR: all 10
#   MRI: PAT-002, PAT-005, PAT-011, PAT-012, PAT-013, PAT-014, PAT-015 (7 of 10)
#   CT:  none
#
# IRB 3: 6 overlap (PAT-002, PAT-005, PAT-007, PAT-010, PAT-011, PAT-016) + 4 new (PAT-017 to PAT-020)
#   EHR: all 10
#   CT:  all 10  (fills gaps for PAT-002, PAT-005, PAT-007, PAT-010, PAT-011, PAT-016)
#   MRI: all 10  (fills gaps for PAT-007, PAT-010, PAT-016, and new patients)

irb_plan = {
    "IRB-2025-001": {
        "patients": [f"PAT-{i:03d}" for i in range(1, 11)],
        "ehr": [f"PAT-{i:03d}" for i in range(1, 11)],
        "ct":  ["PAT-001", "PAT-003", "PAT-004", "PAT-006", "PAT-008", "PAT-009"],
        "mri": [],
        "start_date": datetime(2025, 3, 1),
    },
    "IRB-2025-002": {
        "patients": ["PAT-002", "PAT-005", "PAT-007", "PAT-010",
                      "PAT-011", "PAT-012", "PAT-013", "PAT-014", "PAT-015", "PAT-016"],
        "ehr": ["PAT-002", "PAT-005", "PAT-007", "PAT-010",
                "PAT-011", "PAT-012", "PAT-013", "PAT-014", "PAT-015", "PAT-016"],
        "ct":  [],
        "mri": ["PAT-002", "PAT-005", "PAT-011", "PAT-012", "PAT-013", "PAT-014", "PAT-015"],
        "start_date": datetime(2025, 6, 15),
    },
    "IRB-2025-003": {
        "patients": ["PAT-002", "PAT-005", "PAT-007", "PAT-010", "PAT-011", "PAT-016",
                      "PAT-017", "PAT-018", "PAT-019", "PAT-020"],
        "ehr": ["PAT-002", "PAT-005", "PAT-007", "PAT-010", "PAT-011", "PAT-016",
                "PAT-017", "PAT-018", "PAT-019", "PAT-020"],
        "ct":  ["PAT-002", "PAT-005", "PAT-007", "PAT-010", "PAT-011", "PAT-016",
                "PAT-017", "PAT-018", "PAT-019", "PAT-020"],
        "mri": ["PAT-002", "PAT-005", "PAT-007", "PAT-010", "PAT-011", "PAT-016",
                "PAT-017", "PAT-018", "PAT-019", "PAT-020"],
        "start_date": datetime(2025, 10, 1),
    },
}


### MOCK CONTENT GENERATORS

In [ ]:
# ─── MOCK CONTENT GENERATORS ───

conditions = ["hypertension", "type 2 diabetes", "atrial fibrillation", "COPD",
              "chronic kidney disease", "heart failure", "coronary artery disease",
              "asthma", "hyperlipidemia", "anemia"]

medications = ["lisinopril 10mg", "metformin 500mg", "atorvastatin 20mg",
               "amlodipine 5mg", "metoprolol 50mg", "omeprazole 20mg",
               "levothyroxine 75mcg", "albuterol inhaler", "aspirin 81mg",
               "furosemide 40mg"]

def random_date(start, spread_days=60):
    return start + timedelta(days=random.randint(0, spread_days))

def make_ehr(patient_id, irb_id, date):
    age = random.randint(35, 82)
    sex = random.choice(["Male", "Female"])
    n_cond = random.randint(1, 3)
    n_meds = random.randint(1, 4)
    pat_conditions = random.sample(conditions, n_cond)
    pat_meds = random.sample(medications, n_meds)
    bp_sys = random.randint(110, 165)
    bp_dia = random.randint(65, 95)
    hr = random.randint(58, 102)
    weight = round(random.uniform(55, 120), 1)

    return f"""ELECTRONIC HEALTH RECORD
========================
Patient ID:    {patient_id}
IRB Protocol:  {irb_id}
Record Date:   {date.strftime('%Y-%m-%d')}

DEMOGRAPHICS
  Age: {age}
  Sex: {sex}

ACTIVE CONDITIONS
  {chr(10).join(f'  - {c}' for c in pat_conditions)}

CURRENT MEDICATIONS
  {chr(10).join(f'  - {m}' for m in pat_meds)}

VITALS
  Blood Pressure: {bp_sys}/{bp_dia} mmHg
  Heart Rate:     {hr} bpm
  Weight:         {weight} kg

NOTES
  Patient seen for routine follow-up. Condition stable.
  Continue current treatment plan. Follow up in 3 months.
"""

def make_ct(patient_id, irb_id, date):
    region = random.choice(["chest", "abdomen", "head", "chest/abdomen"])
    slices = random.choice([64, 128, 256])
    thickness = random.choice(["1.0mm", "1.25mm", "2.5mm", "5.0mm"])
    contrast = random.choice(["with IV contrast", "without contrast"])
    finding = random.choice([
        "No acute findings. Stable appearance.",
        "Small pulmonary nodule, 4mm. Recommend follow-up in 12 months.",
        "Mild hepatic steatosis noted. No focal lesions.",
        "Unremarkable study. No evidence of acute pathology.",
        "Mild degenerative changes. No acute abnormality.",
        "Small pleural effusion on the left. Otherwise unremarkable.",
    ])

    return f"""CT IMAGING REPORT
========================
Patient ID:    {patient_id}
IRB Protocol:  {irb_id}
Study Date:    {date.strftime('%Y-%m-%d')}

ACQUISITION
  Region:        {region}
  Slices:        {slices}
  Thickness:     {thickness}
  Contrast:      {contrast}
  Scanner:       Siemens SOMATOM Force

FINDINGS
  {finding}

IMPRESSION
  Reviewed by radiology. Filed under IRB protocol {irb_id}.
"""

def make_mri(patient_id, irb_id, date):
    region = random.choice(["brain", "cardiac", "lumbar spine", "abdomen"])
    tesla = random.choice(["1.5T", "3.0T"])
    sequences = random.choice([
        "T1, T2, FLAIR, DWI",
        "T1, T2, STIR, post-contrast T1",
        "Cine SSFP, T1 mapping, T2 mapping, LGE",
        "T1, T2, DWI, ADC map",
    ])
    finding = random.choice([
        "No acute intracranial abnormality.",
        "Normal cardiac morphology and function. EF estimated at 58%.",
        "Mild disc desiccation at L4-L5. No canal stenosis.",
        "No focal signal abnormality. Unremarkable study.",
        "Small area of white matter hyperintensity, nonspecific. Age-appropriate.",
        "Normal hepatic signal intensity. No focal lesions.",
    ])

    return f"""MRI IMAGING REPORT
========================
Patient ID:    {patient_id}
IRB Protocol:  {irb_id}
Study Date:    {date.strftime('%Y-%m-%d')}

ACQUISITION
  Region:        {region}
  Field Strength:{tesla}
  Sequences:     {sequences}
  Scanner:       GE SIGNA Premier

FINDINGS
  {finding}

IMPRESSION
  Reviewed by radiology. Filed under IRB protocol {irb_id}.
"""


## Generate Files

In [ ]:
# ─── GENERATE FILES ───

generators = {"ehr": make_ehr, "ct": make_ct, "mri": make_mri}
file_count = 0

for irb_id, plan in irb_plan.items():
    irb_dir = os.path.join(base, irb_id.lower().replace("-", "_"))

    # Always create all 3 subdirectories (even if empty)
    for modality in ["ehr", "ct", "mri"]:
        mod_dir = os.path.join(irb_dir, modality)
        os.makedirs(mod_dir, exist_ok=True)

        for patient_id in plan[modality]:
            date = random_date(plan["start_date"])
            content = generators[modality](patient_id, irb_id, date)
            filename = f"{patient_id}_{modality}.txt"
            filepath = os.path.join(mod_dir, filename)
            with open(filepath, "w") as f:
                f.write(content)
            file_count += 1

print(f"Generated {file_count} files.\n")

# ─── PRINT SUMMARY ───
print("=" * 60)
print("MOCK DATA SUMMARY")
print("=" * 60)

for irb_id, plan in irb_plan.items():
    print(f"\n{irb_id}")
    print(f"  Patients: {', '.join(plan['patients'])}")
    print(f"  EHR:      {len(plan['ehr'])} files")
    print(f"  CT:       {len(plan['ct'])} files")
    print(f"  MRI:      {len(plan['mri'])} files")

# Show overlap
irb1_set = set(irb_plan["IRB-2025-001"]["patients"])
irb2_set = set(irb_plan["IRB-2025-002"]["patients"])
irb3_set = set(irb_plan["IRB-2025-003"]["patients"])

print(f"\nOVERLAP")
print(f"  IRB 1 ∩ IRB 2: {sorted(irb1_set & irb2_set)}")
print(f"  IRB 1 ∩ IRB 3: {sorted(irb1_set & irb3_set)}")
print(f"  IRB 2 ∩ IRB 3: {sorted(irb2_set & irb3_set)}")
print(f"  All three:     {sorted(irb1_set & irb2_set & irb3_set)}")
print(f"\n  Unique patients across all IRBs: {len(irb1_set | irb2_set | irb3_set)}")

## Hide some IRBs for now

In [ ]:
# Make subdirectory
dir_hidden = base / "hidden"
dir_hidden.mkdir(exist_ok=True)

# Move IRBs 002 & 003 to hidden
for subdir in ["irb_2025_002", "irb_2025_003"]:
    src = base / subdir
    dst = dir_hidden / subdir
    src.rename(dst)

    # Confirm
    print(f"Moved {src} to {dst}")
